# Month 4 / Final Phase: Live Demo & Model Inference Pipeline

This notebook provides a presentation-ready, end-to-end live demonstration of the final trained **Ensemble Heart Disease Prediction Framework**.

### Key Capabilities Demonstrated:
1. **Artifact Loading**: Loads fitted preprocessor and tuned models (`RandomForest`, `XGBoost`, `AdaBoost`, and `Soft Ensemble`) trained on the full 920-patient combined 4-site UCI cohort.
2. **Multi-Patient Case Walkthroughs**: Analyzes 3 distinct patient profiles (Low Risk, Moderate Risk, High Risk) with full clinical features.
3. **Uncertainty & Consensus Flagging**: Computes per-patient model agreement scores across base classifiers to flag potential prediction uncertainty.
4. **SHAP Local Explainability**: Extracts top-3 risk drivers influencing each prediction.
5. **Clinical Disclaimer**: Formats output clearly with an explicit disclaimer line (*"This is a machine learning estimate, not a medical diagnosis."*).

In [1]:
import sys
import os
import json
import pandas as pd

# Ensure repo root is on sys.path
sys.path.append(os.path.abspath(".."))

from src.preprocessing import load_combined_dataset
from src.predict import load_artifacts, predict_patient_risk, format_summary_output

print("Libraries successfully loaded.")

Libraries successfully loaded.


## 1. Verify Saved Model Artifacts & Metadata

In [2]:
# Load metadata
metadata_path = "../models/metadata.json"
if os.path.exists(metadata_path):
    with open(metadata_path, "r") as f:
        metadata = json.load(f)
    print(f"Training Date    : {metadata.get('training_date')}")
    print(f"Dataset Name     : {metadata.get('dataset_name')}")
    print(f"Dataset Size     : {metadata.get('dataset_size')} patients")
    print(f"Features Count   : {metadata.get('processed_feature_count')}")
    print("\nBase Model Optuna Inner-CV ROC-AUC Scores:")
    for m_name, score in metadata.get("optuna_validation_roc_auc_scores", {}).items():
        print(f"  - {m_name:15s}: {score:.4f}")
else:
    print("Metadata not found. Please run src/train_final_model.py first.")

Training Date    : 2026-09-17T04:16:36.005408+00:00
Dataset Name     : Combined 4-Site UCI Heart Disease Dataset
Dataset Size     : 920 patients
Features Count   : 13

Base Model Optuna Inner-CV ROC-AUC Scores:
  - random_forest  : 0.8806
  - xgboost        : 0.8849
  - adaboost       : 0.8859


## 2. Load Patient Cases for Live Demonstration

In [3]:
# Load combined dataset to pick sample patient rows
df_combined = load_combined_dataset(data_dir="../data", results_dir="../results")

print(f"Loaded combined dataset with {len(df_combined)} rows.")

# Sample representative patient rows from different risk profiles:
# 1. Low Risk Case (Patient Row 50)
# 2. Moderate Risk Case (Patient Row 0)
# 3. High Risk Case (Patient Row 300)
sample_indices = {
    "Case 1: Low-Risk Profile": 50,
    "Case 2: Moderate-Risk Profile": 0,
    "Case 3: High-Risk Profile": 300
}

Loaded combined dataset with 920 rows.


## 3. Run Inference & Generate Patient Risk Summaries

In [4]:
for case_title, row_idx in sample_indices.items():
    patient_data = df_combined.iloc[row_idx].to_dict()
    
    # Run prediction through saved preprocessor and models
    result = predict_patient_risk(patient_data, models_dir="../models")
    
    # Format and print clean clinical summary
    summary_text = format_summary_output(patient_data, result)
    
    print(f"\n>>> {case_title.upper()} (Source Site: {patient_data.get('source_site', 'N/A')}, True Ground Label: {patient_data.get('target', 'N/A')}) <<<")
    print(summary_text)
    print("\n" + "#"*60 + "\n")


>>> CASE 1: LOW-RISK PROFILE (Source Site: cleveland, True Ground Label: 0) <<<
     HEART DISEASE RISK ASSESSMENT SUMMARY       
Predicted Probability : 16.4% (0.1642)
Risk Category         : LOW RISK
--------------------------------------------------
MODEL CONSENSUS & UNCERTAINTY:
  Model agreement: moderate agreement — RF: 12%, XGBoost: 9%, AdaBoost: 28%. Moderate variance among base classifiers.
  Base Classifiers:
    - Random Forest  : 12.0%
    - XGBoost        : 9.0%
    - AdaBoost       : 28.2%
--------------------------------------------------
TOP CONTRIBUTING FEATURES (SHAP Explanation):
  1. cp (decreases risk, SHAP impact: -0.869)
  2. sex (decreases risk, SHAP impact: -0.515)
  3. age (decreases risk, SHAP impact: -0.333)
--------------------------------------------------
This is a machine learning estimate, not a medical diagnosis.

############################################################




>>> CASE 2: MODERATE-RISK PROFILE (Source Site: cleveland, True Ground Label: 0) <<<
     HEART DISEASE RISK ASSESSMENT SUMMARY       
Predicted Probability : 49.7% (0.4967)
Risk Category         : MODERATE RISK
--------------------------------------------------
MODEL CONSENSUS & UNCERTAINTY:
  Model agreement: high agreement — RF: 54%, XGBoost: 45%, AdaBoost: 50%. High consensus among base classifiers.
  Base Classifiers:
    - Random Forest  : 53.5%
    - XGBoost        : 45.0%
    - AdaBoost       : 50.4%
--------------------------------------------------
TOP CONTRIBUTING FEATURES (SHAP Explanation):
  1. cp (decreases risk, SHAP impact: -0.649)
  2. oldpeak (increases risk, SHAP impact: +0.389)
  3. exang (decreases risk, SHAP impact: -0.221)
--------------------------------------------------
This is a machine learning estimate, not a medical diagnosis.

############################################################




>>> CASE 3: HIGH-RISK PROFILE (Source Site: cleveland, True Ground Label: 1) <<<
     HEART DISEASE RISK ASSESSMENT SUMMARY       
Predicted Probability : 82.8% (0.8277)
Risk Category         : HIGH RISK
--------------------------------------------------
MODEL CONSENSUS & UNCERTAINTY:
  Model agreement: moderate agreement — RF: 90%, XGBoost: 88%, AdaBoost: 70%. Moderate variance among base classifiers.
  Base Classifiers:
    - Random Forest  : 90.5%
    - XGBoost        : 87.6%
    - AdaBoost       : 70.2%
--------------------------------------------------
TOP CONTRIBUTING FEATURES (SHAP Explanation):
  1. cp (increases risk, SHAP impact: +0.553)
  2. exang (increases risk, SHAP impact: +0.439)
  3. chol (decreases risk, SHAP impact: -0.349)
--------------------------------------------------
This is a machine learning estimate, not a medical diagnosis.

############################################################



## 4. Custom Single-Patient CLI Demo

Below we simulate a custom new patient input dictionary passed directly into `predict_patient_risk()`.

In [5]:
custom_patient = {
    "age": 62.0,
    "sex": 1.0,           # Male
    "cp": 4.0,            # Asymptomatic chest pain
    "trestbps": 150.0,    # Resting BP
    "chol": 280.0,        # High cholesterol
    "fbs": 1.0,           # Fasting blood sugar > 120
    "restecg": 2.0,       # Ventricular hypertrophy
    "thalach": 120.0,     # Low max heart rate
    "exang": 1.0,         # Exercise induced angina
    "oldpeak": 2.5,       # High ST depression
    "slope": 2.0,         # Flat ST slope
    "ca": 2.0,            # 2 major vessels
    "thal": 7.0           # Reversable defect
}

custom_result = predict_patient_risk(custom_patient, models_dir="../models")
print(format_summary_output(custom_patient, custom_result))

     HEART DISEASE RISK ASSESSMENT SUMMARY       
Predicted Probability : 90.3% (0.9027)
Risk Category         : HIGH RISK
--------------------------------------------------
MODEL CONSENSUS & UNCERTAINTY:
  Model agreement: moderate agreement — RF: 98%, XGBoost: 93%, AdaBoost: 80%. Moderate variance among base classifiers.
  Base Classifiers:
    - Random Forest  : 97.5%
    - XGBoost        : 92.9%
    - AdaBoost       : 80.4%
--------------------------------------------------
TOP CONTRIBUTING FEATURES (SHAP Explanation):
  1. oldpeak (increases risk, SHAP impact: +0.572)
  2. cp (increases risk, SHAP impact: +0.476)
  3. exang (increases risk, SHAP impact: +0.328)
--------------------------------------------------
This is a machine learning estimate, not a medical diagnosis.
